In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# IVA Decomposition of Wavelet Power

## Scope

Run the **Independent Vector Analysis** algorithm on the 4-D wavelet
power tensor and stop. Analyses of the resulting sources / mixing
matrices live in companion notebooks (next steps).

Unlike single-dataset ICA, **IVA decomposes K datasets jointly** while
preserving the dependency between corresponding sources across
datasets. Here each **subject is one dataset**, so IVA yields a set
of components that are *already aligned* across subjects — the kth
source in subject 1 corresponds to the kth source in subjects 2..K.
This removes the permutation ambiguity that single-subject ICA
introduces and is the main motivation for choosing IVA over ICA in a
group analysis.

## Reshape

Each subject's wavelet tensor is flattened so that **channels ×
frequencies** form the observation axis and **time** is the sample
axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)
Per-subject reshape:
         (n_channels, n_freqs, n_times)
         → (n_channels × n_freqs,  n_times)
           ──── features ──────   samples
Stack:   (n_pca,  n_times,  n_subjects)  — IVA input layout (N, T, K)
```

IVA-G requires a **square** mixing matrix per dataset (`N == feature
dim`), so each subject's `(C×F, T)` matrix is first reduced with a
**per-subject PCA** to `N_PCA` components before stacking.

## Pipeline

1. Load wavelet power for the chosen condition / music type (cached).
2. Apply subject / channel / time subsets for fast iteration.
3. Z-score along time.
4. Per-subject reshape to `(C×F, T)` and per-subject PCA to `N_PCA`.
5. Stack into `(N_PCA, T, K)` and run **IVA-G**.
6. Recover per-subject sources and per-subject component patterns in
   the original `(F, C)` feature space.

After the final cell the following variables are available for any
downstream analysis notebook:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_subjects` | `(S, C×F, T)` | Per-subject z-scored reshape |
| `pcas` | list[`PCA`] | Per-subject fitted PCA objects |
| `X_pca` | `(N_PCA, T, S)` | PCA-reduced IVA input layout |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing matrices |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `iva_sources_pca` | `(S, N_PCA, T)` | Per-subject IVA sources in PCA space |
| `iva_components` | `(S, N_PCA, F, C)` | IVA component patterns reshaped to (freq, channel) per subject |

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── IVA settings ──────────────────────────────────────────────
N_COMPONENTS_PCA = 50  # per-subject PCA dim before IVA (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 1024
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = True
IVA_RANDOM_STATE = 42  # seeds per-subject PCA + W_init

# ── Storage directory ─────────────────────────────────────────
# Mirror the 04 cache so wavelets are not recomputed when iterating on IVA.
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Per-subject PCA dim    : {N_COMPONENTS_PCA}")
print(f"IVA optimisation       : {IVA_OPT_APPROACH}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/` (shared with the 04 ICA notebooks).

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types. The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Per-Subject Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time
series to zero mean and unit variance so that IVA is not dominated by
high-power channels, subjects, or frequencies.

**Reshape** is done independently per subject: for each subject `k`
the `(C, F, T)` slice becomes a `(C×F, T)` matrix — channels and
frequencies form the observation axis, time stays as samples. The
result is stacked as a `(S, C×F, T)` array (Python view; IVA wants
axis-ordering `(N, T, K)` which we build in the next step).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power |
| `X_subjects` | `(S, C×F, T)` | Per-subject z-scored reshape |

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Per-subject reshape: (S, C, F, T) → (S, C*F, T)
n_feat = n_channels * n_freqs
X_subjects = bb_z.reshape(n_subjects, n_feat, n_times)  # (S, C*F, T)

print(f"Per-subject reshape    : {X_subjects.shape}  (subjects, C*F, time)")
print(f"  Features per subject : {n_feat}  (C={n_channels} * F={n_freqs})")
print(f"  Samples per subject  : {n_times}")

---
## Step 2 — Per-Subject PCA Dimensionality Reduction

`iva_g` assumes a **square** mixing matrix per dataset, i.e. the
number of estimated sources equals the input dimensionality. Running
IVA directly on `(C×F)` features per subject would be prohibitively
expensive (and rank-deficient if `C×F > T`). Each subject's matrix
is reduced with its **own** PCA to `N_PCA` components, after which the
K subject matrices are stacked into IVA's `(N, T, K)` layout.

Keeping per-subject PCAs (rather than a single shared PCA) preserves
subject-specific spatial / spectral subspaces, which is exactly what
IVA exploits to align cross-subject sources.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pcas[k]` | — | Fitted PCA object for subject `k` |
| `pca_evr` | `(S, N_PCA)` | Explained-variance ratio per subject |
| `X_pca` | `(N_PCA, T, S)` | IVA input layout (N, T, K) |

In [ ]:
# Per-subject PCA. PCA expects (n_samples, n_features); each subject's matrix
# is (C*F, T) → transpose to (T, C*F), fit, transform back to (T, N_PCA),
# transpose to (N_PCA, T), then stack along axis 2 as IVA expects.
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_times))
pca_evr = np.zeros((n_subjects, N_COMPONENTS_PCA))

for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (T, C*F) — samples x features for sklearn
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)  # (T, N_PCA)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T  # (N_PCA, T)
    pca_evr[k] = pca.explained_variance_ratio_
    print(
        f"  S{k + 1}: explained variance = {pca_evr[k].sum() * 100:5.1f}% "
        f"({N_COMPONENTS_PCA} comps)"
    )

# IVA expects (N, T, K)
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

print(f"\nIVA input shape        : {X_pca.shape}  (N_PCA, T, K=subjects)")

# Quick visual of how much variance each subject's PCA retains.
fig, ax = plt.subplots(figsize=(8, 4))
for k in range(n_subjects):
    ax.plot(
        np.arange(1, N_COMPONENTS_PCA + 1),
        np.cumsum(pca_evr[k]),
        marker="o",
        markersize=3,
        label=f"S{k + 1}",
    )
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title(f"Per-Subject PCA — {LABEL}")
ax.legend(fontsize=8, ncol=2)
ax.axhline(0.9, ls="--", lw=0.6, color="gray")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 3 — Run IVA-G

`iva_g` returns a demixing matrix `W` of shape `(N, N, K)`. The
sources for subject `k` are then

```
S_pca[k]  = W[:, :, k] @ X_pca[:, :, k]                 # (N_PCA, T)
S_full[k] = pcas[k].components_.T @ S_pca[k]             # (C*F, T)
components[k] = (W[:, :, k] @ pcas[k].components_)       # (N_PCA, C*F)
             .reshape(N_PCA, F, C)
```

Because IVA's permutation ambiguity is **shared across datasets**, the
kth source in `S_full[0]` corresponds to the kth source in
`S_full[1..K-1]` — there is no need to align sources across subjects
post-hoc.

| Returned by `iva_g` | Shape | Description |
|---------------------|-------|-------------|
| `W` | `(N_PCA, N_PCA, S)` | Per-subject demixing matrix |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `Sigma_N` | `(S, S, N_PCA)` | Per-source-component covariance across subjects |
| `isi` | `float` | Joint inter-symbol-interference (only when ground-truth `A` is given) |

In [ ]:
# Deterministic W initialisation: random matrix per subject seeded by IVA_RANDOM_STATE.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))

W, cost, Sigma_N, isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

print(f"\nW shape           : {W.shape}  (N_PCA, N_PCA, K=subjects)")
print(f"Sigma_N shape     : {Sigma_N.shape}  (K, K, N_PCA)")
print(f"Iterations        : {len(cost)}")
print(f"Final cost        : {cost[-1]:.6f}")

# Cost-curve sanity check.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cost, marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("IVA cost")
ax.set_title(f"IVA-G Convergence — {LABEL}")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 4 — Recover Sources and Component Patterns

Apply the per-subject demixing matrix to the PCA-reduced data to
obtain IVA sources, then combine `W_k` with the PCA loadings to
express each component pattern over the original `(F, C)` axes
(directly comparable to `components_2d` in the 04 ICA notebook).

The kth row of `iva_components[k]` is the unmixing vector for the
kth source in subject `k`'s original feature space. Because IVA
resolves the permutation jointly across datasets, the kth row is
**already aligned** across subjects.

In [ ]:
iva_sources_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_times))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_freqs, n_channels))

for k in range(n_subjects):
    W_k = W[:, :, k]  # (N_PCA, N_PCA)
    X_pca_k = X_pca[:, :, k]  # (N_PCA, T)

    # Sources in the PCA subspace: (N_PCA, T)
    iva_sources_pca[k] = W_k @ X_pca_k

    # Project the unmixing rows back through the per-subject PCA loadings to
    # express component patterns in the full (C*F) feature space.
    components_full = W_k @ pcas[k].components_  # (N_PCA, C*F)

    # X_subjects was reshape (S, C, F, T) → (S, C*F, T), so the flattened axis
    # iterates channels (slow) then frequencies (fast). Reshape back to
    # (N_PCA, C, F) and transpose to (N_PCA, F, C) to mirror the 04 ICA
    # notebook's components_2d layout.
    iva_components[k] = components_full.reshape(
        N_COMPONENTS_PCA, n_channels, n_freqs
    ).transpose(0, 2, 1)

print(f"IVA sources (PCA space)  : {iva_sources_pca.shape}   (S, N_PCA, T)")
print(f"IVA components (F, C)    : {iva_components.shape}    (S, N_PCA, F, C)")

---
## Result Summary

All variables required for downstream analyses are now in memory:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power tensor |
| `X_subjects` | `(S, C×F, T)` | Per-subject z-scored reshape |
| `pcas` | list[`PCA`] | Per-subject fitted PCA objects |
| `pca_evr` | `(S, N_PCA)` | Per-subject PCA explained-variance ratio |
| `X_pca` | `(N_PCA, T, S)` | IVA input (PCA-reduced) |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing |
| `Sigma_N` | `(S, S, N_PCA)` | Per-SCV covariance across subjects |
| `cost` | `(n_iter,)` | IVA convergence trace |
| `iva_sources_pca` | `(S, N_PCA, T)` | Sources in PCA subspace |
| `iva_components` | `(S, N_PCA, F, C)` | Component patterns reshaped to (freq, channel) |

The kth source/component is **aligned across subjects** by
construction — no post-hoc matching is required. Downstream
notebooks can now compute ISC, topomaps, frequency profiles, etc.
on top of these variables.